In [ ]:
import pandas as pd
import tensorflow as tf

tf.config.threading.set_intra_op_parallelism_threads(0)
tf.config.threading.set_inter_op_parallelism_threads(0)
tf.config.optimizer.set_jit(True)

from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


In [2]:
df = pd.read_csv("../data/processed/dia_flights.csv")
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442164 entries, 0 to 442163
Data columns (total 30 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Departure delay (Minutes)  442164 non-null  float64
 1   Arrival Delay (Minutes)    442164 non-null  float64
 2   temp                       442164 non-null  float64
 3   dwpt                       442164 non-null  float64
 4   rhum                       442164 non-null  float64
 5   prcp                       442164 non-null  float64
 6   wdir                       442164 non-null  float64
 7   wspd                       442164 non-null  float64
 8   pres                       442164 non-null  float64
 9   monthly_passenger_arr      442164 non-null  int64  
 10  monthly_freight_arr        442164 non-null  int64  
 11  total_arr                  442164 non-null  int64  
 12  monthly_seats_arr          442164 non-null  int64  
 13  monthly_passenger_dep      44

In [3]:
y_scaler = StandardScaler()
x_scaler = StandardScaler()
target = df["Departure delay (Minutes)"].values.astype(float)

feature_df = df.drop(columns=["15min_delay", "Departure delay (Minutes)"])
x = feature_df.values.astype(float)  # directly to ndarray

x_scaled= x_scaler.fit_transform(x)
y_scaled = y_scaler.fit_transform(target.reshape(-1,1))

#sliding window for LSTM
window = 12
x_seq = []
y_seq = []

for i in range(window,len(x_scaled)):
    x_seq.append(x[i-window:i])
    y_seq.append(y_scaled[i])

x_seq = np.array(x_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.float32)

#split somewhat different, can't mix temporal data
split = int(0.8*len(x_seq))
x_train = x_seq[:split]
x_test = x_seq[split:]
y_train = y_seq[:split]
y_test = y_seq[split:]


model = Sequential([
    LSTM(64, return_sequences = True, input_shape=(window, x_train.shape[2])),
    Dropout(0.1),
    LSTM(32),
    Dropout(0.1),
    Dense(16, activation='relu'),
    #for binary classification vv
    Dense(1, activation='linear') 
])


model.compile(
    loss="mse",
    optimizer=Adam(learning_rate=0.001),
    metrics=["mae","mse"]
)


c:\Users\ivanl\miniconda3\envs\flight_delay_predictor\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [4]:
history = model.fit(
    x_train,y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=64,
    shuffle=False,
    verbose=1
    )

Epoch 1/20
4975/4975 ━━━━━━━━━━━━━━━━━━━━ 32s 6ms/step - loss: 1.0389 - mae: 0.5190 - mse: 1.0389 - val_loss: 1.4296 - val_mae: 0.6105 - val_mse: 1.4296
Epoch 2/20
4975/4975 ━━━━━━━━━━━━━━━━━━━━ 28s 6ms/step - loss: 1.0385 - mae: 0.5172 - mse: 1.0385 - val_loss: 1.4266 - val_mae: 0.6221 - val_mse: 1.4266
Epoch 3/20
4975/4975 ━━━━━━━━━━━━━━━━━━━━ 28s 6ms/step - loss: 1.0392 - mae: 0.5189 - mse: 1.0392 - val_loss: 1.4273 - val_mae: 0.6188 - val_mse: 1.4273
Epoch 4/20
4975/4975 ━━━━━━━━━━━━━━━━━━━━ 29s 6ms/step - loss: 1.0401 - mae: 0.5201 - mse: 1.0401 - val_loss: 1.4272 - val_mae: 0.6192 - val_mse: 1.4272
Epoch 5/20
4975/4975 ━━━━━━━━━━━━━━━━━━━━ 29s 6ms/step - loss: 1.0385 - mae: 0.5182 - mse: 1.0385 - val_loss: 1.4271 - val_mae: 0.6193 - val_mse: 1.4271
Epoch 6/20
4975/4975 ━━━━━━━━━━━━━━━━━━━━ 29s 6ms/step - loss: 1.0394 - mae: 0.5204 - mse: 1.0394 - val_loss: 1.4266 - val_mae: 0.6222 - val_mse: 1.4266
Epoch 7/20
4975/4975 ━━━━━━━━━━━━━━━━━━━━ 28s 6ms/step - loss: 1.0398 - mae: 0.519

In [8]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
y_pred_scaled = model.predict(x_test)
y_pred = y_scaler.inverse_transform(y_pred_scaled)
y_test_true = y_scaler.inverse_transform(y_test)
mae = mean_absolute_error(y_test_true, y_pred)
rmse = root_mean_squared_error(y_test_true, y_pred)

print(f"MAE (minutes): {mae:.2f}")
print(f"RMSE (minutes): {rmse:.2f}")

2764/2764 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step
MAE (minutes): 19.50
RMSE (minutes): 30.99


In [ ]:
import joblib

model.save("../models/delay_reg_lstm.keras")

joblib.dump(x_scaler, "../models/reg_x_scaler.pkl")
joblib.dump(y_scaler, "../models/reg_y_scaler.pkl")

['../models/reg_y_scaler.pkl']